In [ ]:
%%capture
%pip install keybert==0.9.0 bitsandbytes==0.49.2

In [ ]:
import os, torch, logging

import polars as pl

from dotenv import load_dotenv

import plotly.io as pio
import plotly.express as px

from transformers import BitsAndBytesConfig
from sentence_transformers import SentenceTransformer

from bertopic import BERTopic
from bertopic.representation import MaximalMarginalRelevance

from transformers import logging as hf_logging
from transformers.utils.logging import disable_progress_bar

In [ ]:
# Configure data, models, device, quantization config, and plot configurations
DATA_PATH = "../../data/train.csv"
OUTPUT_PLOT = "../../plots/frequent_topic_counts.png"

OPTION_COLS = ["A", "B", "C", "D", "E"]

data = pl.read_csv(DATA_PATH)

data = data.with_columns(
    (
        pl.lit("Prompt : ") + pl.col("prompt")
        + pl.lit(" Options: ")
        + pl.concat_str(
            [pl.lit(f"{opt}) ") + pl.col(opt).fill_null(" ") for opt in OPTION_COLS],
            separator=" "
        )
    ).alias("mcq_query")
)
docs = data["mcq_query"].fill_null("").to_list()

EMBED_MODEL = "zeroentropy/zembed-1-embedding"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
    llm_int8_has_fp16_weight=False
)

COL1 = '#00040A'
COL2 = '#202124'
COL3 = '#E1E1E0'
GRID = '#525458'

TITLE_FONT_SIZE = 20
LABEL_FONT_SIZE = 16
TICK_FONT_SIZE = 12

PLOT_WDTH = 1150
PLOT_HGHT = 800
pio.renderers.default = "notebook"

# Configure HuggingFace API key
load_dotenv()
os.environ["HF_TOKEN"] = os.getenv("HF_READ_TOKEN") if os.getenv("HF_READ_TOKEN") else "" # type: ignore

# Configure logging levels to hide model-loading report
hf_logging.set_verbosity_error()

logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)

disable_progress_bar()

In [ ]:
# Get topic labels present in train data
embed_model = SentenceTransformer(
    EMBED_MODEL,
    trust_remote_code=True,
    model_kwargs={
        "quantization_config": bnb_config, 
        "dtype": torch.float16
    }
).to(DEVICE)

embeddings = embed_model.encode(docs, show_progress_bar=True)

representation_model = MaximalMarginalRelevance(diversity=0.3)

topic_model = BERTopic(
    embedding_model=embed_model,
    representation_model=representation_model,
    min_topic_size=15, 
    verbose=True
)

topics, probs = topic_model.fit_transform(docs, embeddings) # type: ignore

topic_info = topic_model.get_topic_info()
topic_mapping = dict(zip(topic_info["Topic"], topic_info["Name"]))

assigned_labels = [topic_mapping[t] for t in topics]
data = data.with_columns(pl.Series("topic", assigned_labels))

In [ ]:
plot_topic_data = (
    data.group_by("topic")
    .len()
    .sort("len", descending=True)
    .filter(~pl.col("topic").str.starts_with("-1_")) 
    .head(20)
)

In [ ]:
# Clean topic labels from BERTopic to human-readable labels
clean_topic_labels = {
    "0_phenomenon_snapshots_scaling_hysteresis": "Condensed Matter & Non-linear Dynamics",
    "1_hypercrystalline_permanent_switching_subjected": "Memristive Materials & Solid State",
    "2_butterfly_causality_mammals_effect": "Chaos Theory & Dynamical Systems",
    "3_eigenstate_sinusoidal_elementary_poles": "Quantum Mechanics & Wave Theory",
    "4_processes_shower_utilization_outside": "Thermodynamics & Fluid Dynamics",
    "5_metric_loosening_bolts_dominant": "Applied Classical Mechanics",
    "6_holes_fusion_electrodes_hawking": "Astrophysics & Black Hole Physics",
    "7_volunteers_analysis_heisenberg_uncertainty": "Quantum Physics & Uncertainty Principle",
    "8_higher_blocking_antiquarks_antiferromagnetic": "Particle Physics & Magnetism",
    "9_inverse_measurement_symbol_hammer": "Fluid Mechanics & Physical Measurement",
    "10_breaking_symmetry_vector_lorenz": "Theoretical Physics & Relativity",
    "11_noise_optical_osnr_quality": "Optics & Signal Processing",
    "12_petroleum_graduated_hydrometer_penrose": "Fluid Density & Gravitational Physics",
    "13_geological_radiometric_20th_xnav": "Geophysics & Pulsar Navigation",
    "14_mainly_magnetization_caused_irregularity": "Electromagnetism & Magnetics",
    "15_rare_recycling_dry_fresnel": "Optics & Resource Sustainability",
    "16_coffee_atomristor_memristive_grounds": "Nanotechnology & Surface Physics",
    "17_heidegger_atmosphere_believes_existence": "Philosophy & Atmospheric Science",
    "18_metagenomes_microbiome_noether_momentum": "Microbiology & Theoretical Physics",
    "19_conductor_spacetime_magnitude_geodetic": "General Relativity & Astrophysics"
}

plot_topic_data = plot_topic_data.with_columns(
    pl.col("topic").replace(clean_topic_labels).alias("topic")
)

In [ ]:
# frequent_topic_counts
# Plot bar graph with the top 20 most frequenty occuring types of topics
fig = px.bar(
    plot_topic_data, 
    x="len", 
    y="topic",
    color="topic",
    orientation='h',
    labels={
        "topic": "Domain Name", 
        "len": "Number of Questions"
    },
    text_auto=True,
    color_discrete_sequence=px.colors.qualitative.Light24
)

fig.update_layout(
    width=PLOT_WDTH,
    height=PLOT_HGHT,
    paper_bgcolor=COL1,
    plot_bgcolor=COL1,
    font=dict(color=COL3),
    showlegend=False,

    title=dict(
        text="Top 20 MCQ Domains",
        font=dict(color=COL3, size=TITLE_FONT_SIZE),
        x=0.5, xanchor="center"
    ),

    scene=dict(
        bgcolor=COL1,
        xaxis=dict(
            color=COL3,
            gridcolor=GRID,
            gridwidth=1,
            backgroundcolor=COL1
        ),
        yaxis=dict(
            color=COL3,
            gridcolor=GRID,
            gridwidth=1,
            backgroundcolor=COL1
        )
    )
)

fig.write_image(OUTPUT_PLOT)
fig.show()